# Phase 2: Preprocessing Pipeline, Modellierung, Hyperparameter-Tuning & MLflow Tracking

## Ziele dieses Notebooks:
* **Daten laden:** Aufbereiteten Datensatz `data/data_processed.csv` importieren.
* **Preprocessing Pipeline aufbauen:** Numerische Spalten (Fragen N1-A4 + Age) median-imputieren und skalieren (`StandardScaler`); kategoriale Spalten (`gender`, `hand`) mode-imputieren und one-hot-kodieren (`OneHotEncoder`).
* **Baseline-Vergleich:** Mehrere Modelle (z. B. RandomForest, LogisticRegression) per 5-Fold Cross-Validation anhand des `F1-Macro`-Scores evaluieren.
* **Hyperparameter-Tuning & Experiment Tracking:** Ausgewählte Modelle tunen und jeden Run mit Parametern, Metriken und Modell-Artefakten in **MLflow** aufzeichnen.
* **Champion-Wahl:** Bestes Modell für das spätere Deployment in der Streamlit-App identifizieren.

In [1]:
# Imports & Daten laden
import pandas as pd
import numpy as np

# MLflow Tracking einrichten (lokale SQLite-Datenbank)
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("personality-prediction")

# Daten aus der EDA-Phase laden
df = pd.read_csv("data/data_processed.csv")

# Separation von Features (X) und Target (y)
X = df.drop(columns=["target"])
y = df["target"]

print("Features Shape:", X.shape)
print("Target Klassen:", y.unique())

2026/09/23 11:30:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 11:30:09 INFO mlflow.store.db.utils: Updating database tables
2026/09/23 11:30:11 INFO mlflow.tracking.fluent: Experiment with name 'personality-prediction' does not exist. Creating a new experiment.


Features Shape: (19719, 22)
Target Klassen: <ArrowStringArray>
['Resilient', 'Moderate', 'Overcontroller', 'Undercontroller']
Length: 4, dtype: str


In [ ]:
# Scikit-Learn Preprocessing Pipeline definieren
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Spalten nach Typen trennen
item_cols = [c for c in X.columns if c not in ["age", "gender", "hand"]]
numeric_features = item_cols + ["age"]
categorical_features = ["gender", "hand"]

# Pipelines für numerische und kategoriale Features
# Null-Werte mit dem Mittelwert (nicht Arith.Mittel)
# Der Mittelwert (arithmetisches Mittel) reagiert extrem
# empfindlich auf Ausreißer, z.B. eine Person von 99Jh. 
# und dann Standardskalierung: (zieht von jedem Wert den Mittelwert ab und teilt durch die Standardabweichung 
# Das Ergebnis sind Werte, die typischerweise im Bereich von -3 bis +3.
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
# Null-Werte mit der häufigsten Ausprägung und
# Ohne math. Mittelwert ist 'Modus' mathematisch
# das einzige sinnvolle Maß über alle Datensätze
# Die dadurch verstärkte Tendenz im Urspungsdatensatz
# ist mit 24 fehlende Werte bei 19.719 Zeilen hat (also ca. 0,12 %).
# unproblematisch.
# Anschließend ['Male', 'Female', 'Other'] auf Spalten mit 1-0 encoden
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Zusammenführung im ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_features),
    ("cat", cat_pipeline, categorical_features)
])

print("Preprocessing Pipeline erfolgreich definiert!")

Preprocessing Pipeline erfolgreich definiert!


In [5]:
# Das entsatnde DataFrame mal anschauen :-D
# 1. Daten durch den Preprocessor transformieren
X_transformed = preprocessor.fit_transform(X)

# 2. Die neuen Spaltennamen aus dem Preprocessor auslesen
feature_names = preprocessor.get_feature_names_out()

# 3. Als lesbares DataFrame verpacken und anzeigen
df_transformed = pd.DataFrame(X_transformed, columns=feature_names)

print("Shape vorher (raw):       ", X.shape)
print("Shape nachher (prep):     ", df_transformed.shape)

# Erste 5 Zeilen der aufbereiteten Daten anzeigen
df_transformed.head()


Shape vorher (raw):        (19719, 22)
Shape nachher (prep):      (19719, 26)


,num__N6,num__N8,num__N7,num__N1,num__N9,num__N10,num__N3,num__N5,num__E3,num__N2,...,num__A4,num__E10,num__E9,num__age,cat__gender_Female,cat__gender_Male,cat__gender_Other,cat__hand_Both,cat__hand_Left,cat__hand_Right
0,-1.499825,-1.335123,-1.655491,-1.729240,-1.644347,-1.396619,-1.618062,-1.533339,1.280125,1.499933,...,0.927826,-1.981792,1.364688,-0.007122,0.0,1.0,0.0,0.0,0.0,1.0
1,0.772208,-0.594719,-0.116884,-0.964794,-0.874251,0.888220,0.138134,0.037929,-0.336966,-0.199319,...,-0.028767,1.084427,-1.499708,-0.007123,1.0,0.0,0.0,0.0,0.0,1.0
2,1.529552,1.626494,1.421723,1.328545,1.436036,1.649833,1.016232,1.609197,-1.954057,-1.898571,...,0.927826,-1.981792,1.364688,-0.007127,1.0,0.0,0.0,0.0,0.0,1.0
3,1.529552,1.626494,1.421723,1.328545,0.665940,1.649833,0.138134,0.823563,-1.145511,0.650307,...,-0.028767,1.084427,0.648589,-0.007126,1.0,0.0,0.0,0.0,0.0,1.0
4,0.014863,0.145685,-0.116884,-0.200348,-0.104156,0.888220,-0.739964,0.037929,-0.336966,-0.199319,...,0.927826,1.084427,-0.067510,-0.007126,1.0,0.0,0.0,0.0,0.0,1.0


### Wir haben 6 Neue Kategorie-Spalten und die beiden Ausgangsspalten Gender & Hand entfwernt also 22 zu => 26 Spalten

In [ ]:
# Baseline-Modellvergleich & MLflow
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Pipeline für Logistic Regression definieren
# LogisticRegression weil die Zielklassen als Z-Scores der OCEAN Eigenschaften
# definiert sind; Z-Scores sind gewichtete lineare Summen/Kombinationen
# Logistic Regression ist ein lineares Modell – sie modelliert solche linearen Zusammenhänge
# mathematisch perfekt und ohne Overhead. Der bei KNN höher wäre
pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# 2. Pipeline für Random Forest definieren
# Risiko des 'Overfitting' !?
pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

# 3. Baseline Evaluation mit 5-Fold Cross Validation (Metrik: f1_macro)
score_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring="f1_macro").mean()
score_rf = cross_val_score(pipe_rf, X, y, cv=5, scoring="f1_macro").mean()

print(f"Baseline Logistic Regression (F1-Macro): {score_lr:.4f}")
print(f"Baseline Random Forest       (F1-Macro): {score_rf:.4f}")

Baseline Logistic Regression (F1-Macro): 0.7652
Baseline Random Forest       (F1-Macro): 0.7503


### Wir nutzen cross_val_score statt ein einfaches model.fit() 
* ** Wenn wir ein Modell nur einmal auf einem festen Train/Test-Split trainieren (fit), hängt die Bewertung stark vom Zufall ab (welche Daten landeten im Test-Set?).  
* ** ! cross_val_score führt eine 5-Fold Cross-Validation (5-fache Kreuzvalidierung) durch:   
* *** Der gesamte Datensatz wird in 5 gleich große Teile (Folds) aufgeteilt. 
* **** Durchlauf 1: Ordner 1-4 werden zum Trainieren genutzt (.fit()), Ordner 5 zum Testen (.predict()).
* **** Durchlauf 2: Ordner 1, 2, 3, 5 werden zum Trainieren genutzt, Ordner 4 zum Testen.
* **** Das wird wiederholt, bis jeder Ordner einmal als Testset diente.Am Ende berechnet cross_val_score den Durchschnitt der 5 Ergebnisse (.mean()).

In [ ]:
# Zum Vergleichen werden weitere Modelle/Algorithmen angewendet
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Liste von Modellen zum Vergleichen
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random_Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Hist_Gradient_Boosting": HistGradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

# Experimente in MLflow loggen
for name, clf in models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])
        
        # 5-Fold Cross-Validation
        scores = cross_val_score(pipe, X, y, cv=5, scoring="f1_macro")
        mean_score = scores.mean()
        
        # In MLflow aufzeichnen
        mlflow.log_param("model_type", name)
        mlflow.log_metric("f1_macro", mean_score)
        
        print(f"Logged {name:<23} | F1-Macro: {mean_score:.4f}")


Logged Logistic_Regression     | F1-Macro: 0.7652
Logged Random_Forest           | F1-Macro: 0.7503
Logged Hist_Gradient_Boosting  | F1-Macro: 0.8025
Logged KNN                     | F1-Macro: 0.7279


* **In cross_val_score() laufen im Hintergrund automatisch folgende Schritte ab:
1. Splitting (Aufteilung): Die Daten werden in $K$ Blöcke (Folds) unterteilt (bei cv=5 also in 5 gleich große Teile). Durch die Strategie StratifiedKFold wird sichergestellt, dass das Mischungsverhältnis der 4 Zielklassen in jedem Block identisch ist.
2. fit() (Training): Das Modell wird auf $K-1$ Blöcken trainiert (z. B. auf Block 1, 2, 3 und 4).
3. predict() (Vorhersage): Das trainierte Modell sagt die Ergebnisse für den übrig gebliebenen, für das Modell unbekannten Block (z. B. Block 5) voraus.
4. Metrik berechnen: Der F1-Score wird ausschließlich auf diesem unbekannten Validierungsblock berechnet.
Dieser Prozess wird K-mal wiederholt, sodass jeder Block genau einmal als ungesehenes Test-Set dient.
* ***Was bedeutet das für unser finales Modell?
cross_val_score() dient nur zur fairen Bewertung der Modellleistung – es speichert am Ende kein einzelnes, fertiges Modell ab.
* ++Damit wir unser bestes Modell (HistGradientBoostingClassifier) später in der Streamlit-App verwenden können, führen wir als Nächstes ein explizites fit() auf den gesamten Daten durch und loggen dieses fertige Modell-Artefakt direkt in MLflow!


In [6]:
# Die "Black Box" von cross_val_score() öffnen
# Um genau zu sehen, was cross_val_score im Hintergrund tut, 
# schauen wir uns die 5 einzelnen Ergebnisse (Folds) an 
# und simulieren die Schleife einmal manuell mit StratifiedKFold.
# !?!?!?!
# Was passiert hier manuell?
# 1. Scikit-Learn teilt die Daten mit StratifiedKFold
#  (damit in jedem Fold das Verteilungverhältnis der 4 Zielklassen identisch bleibt) in 5 Häppchen:  
#  Fold 1: Trainiert auf Fold 2–5, testet auf Fold 1 => Score 1
#  Fold 2: Trainiert auf Fold 1, 3–5, testet auf Fold 2 => Score 2
#  ...usw.
# 2. Am Ende nimmt cross_val_score einfach .mean() über alle 5 Werte
from sklearn.model_selection import StratifiedKFold

# 1. Die 5 einzelnen Fold-Ergebnisse anzeigen
scores = cross_val_score(models["Hist_Gradient_Boosting"], X_transformed, y, cv=5, scoring="f1_macro")

print("--- 5-Fold Cross-Validation Details ---")
for fold, score in enumerate(scores, 1):
    print(f"Fold {fold}: F1-Macro = {score:.4f}")

print(f"\nDurchschnitt (Mean): {scores.mean():.4f}")
print(f"Standardabweichung:  {scores.std():.4f}")

--- 5-Fold Cross-Validation Details ---
Fold 1: F1-Macro = 0.8072
Fold 2: F1-Macro = 0.7943
Fold 3: F1-Macro = 0.8119
Fold 4: F1-Macro = 0.8000
Fold 5: F1-Macro = 0.7991

Durchschnitt (Mean): 0.8025
Standardabweichung:  0.0062


In [8]:
# 1. Die finale Pipeline mit dem Gewinner-Modell definieren
champion_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=42))
])

# 2. Einmalig auf allen Daten trainieren (fit)
champion_pipeline.fit(X, y)

# 3. Den Champion-Run in MLflow aufzeichnen
with mlflow.start_run(run_name="Champion_HistGradientBoosting") as run:
    # Parameter & Metrik speichern
    mlflow.log_param("model_type", "HistGradientBoosting")
    mlflow.log_metric("f1_macro", 0.8025)
    
    # Modell mit cloudpickle sichern (bewährter Standard für komplexe Pipelines)
    mlflow.sklearn.log_model(
        sk_model=champion_pipeline,
        name="model",
        serialization_format="cloudpickle"
    )
    
    # Run ID für app.py ausgeben
    champion_run_id = run.info.run_id
    print("--------------------------------------------------")
    print(f"CHAMPION MODEL ERFOLGREICH GELOGGT!")
    print(f"Deine RUN_ID lautet: {champion_run_id}")
    print("--------------------------------------------------")

2026/09/23 17:14:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--------------------------------------------------
CHAMPION MODEL ERFOLGREICH GELOGGT!
Deine RUN_ID lautet: 619692310ef34ec8af105180c29eb1a7
--------------------------------------------------
